# 1 · Primeiros passos com os dados do DATASUS

**Para quem é:** profissionais de saúde que nunca programaram.
**O que você vai fazer:** instalar a biblioteca, descobrir quais dados existem e
baixar o seu primeiro conjunto — tudo em poucos minutos.
**Tempo estimado:** 3 a 5 minutos.

> Este notebook roda no **Google Colab** sem instalar nada no seu computador.
> Clique em cada célula na ordem e aperte `Shift + Enter` para executar.

## Passo 1 — Instalar as bibliotecas

A **PySUS** é a biblioteca que conversa com os servidores do DATASUS. Ela é
mantida pelo grupo AlertaDengue (Fiocruz) e é o que faz o download dos arquivos
para você.

Junto dela instalamos a **nest_asyncio**, que é indispensável: por dentro, a
PySUS usa um recurso do Python que entra em conflito com a forma como o
notebook funciona. Sem essa segunda biblioteca, qualquer consulta falha com a
mensagem *"asyncio.run() cannot be called from a running event loop"*.

Na primeira vez a instalação leva um ou dois minutos; depois fica pronta.

In [2]:
%pip install pysus==2.10.6 nest_asyncio -q

Note: you may need to restart the kernel to use updated packages.


### Se o Colab pedir para reiniciar

Depois de instalar, o Colab às vezes mostra um aviso pedindo para **reiniciar a
sessão**. Se aparecer, clique em *Reiniciar sessão* e execute novamente a partir
da próxima célula — não é preciso instalar de novo.

## Passo 2 — Preparar o notebook

Esta célula precisa ser executada **uma vez por sessão**, antes de qualquer
consulta. É ela que faz a PySUS funcionar dentro do notebook.

In [3]:
import nest_asyncio

nest_asyncio.apply()

import pysus

print("Tudo pronto. PySUS versão:", pysus.__version__)

Tudo pronto. PySUS versão: 2.10.6


## Passo 3 — Descobrir o que existe antes de baixar

Este é o hábito mais importante de todos. Os arquivos do DATASUS **não são
uniformes**: uma base pode ter todos os meses de um estado e faltar meses de
outro. Se você pedir um arquivo que não existe, a biblioteca devolve uma tabela
**vazia, sem mensagem de erro** — e é fácil achar que "não houve casos" quando
na verdade o arquivo nunca esteve lá.

A função `list_files` mostra o catálogo **sem baixar nada**. Vamos ver o que
existe do **CNES** (Cadastro Nacional de Estabelecimentos de Saúde) no Acre,
em 2024:

In [4]:
from pysus import list_files

catalogo = list_files(dataset="cnes", state="AC", year=2024)

print(f"Arquivos disponíveis: {len(catalogo)}")
catalogo.head()

Arquivos disponíveis: 134


,name,path,dataset,group,year,month,state,modify
0,public\data\ftp\cnes\LT\2024\01\AC\LTAC2401.pa...,public\data\ftp\cnes\LT\2024\01\AC\LTAC2401.pa...,cnes,None,2024,1,AC,2024-02-15 14:25:00
1,public\data\ftp\cnes\LT\2024\02\AC\LTAC2402.pa...,public\data\ftp\cnes\LT\2024\02\AC\LTAC2402.pa...,cnes,None,2024,2,AC,2024-03-14 07:09:00
2,public\data\ftp\cnes\LT\2024\03\AC\LTAC2403.pa...,public\data\ftp\cnes\LT\2024\03\AC\LTAC2403.pa...,cnes,None,2024,3,AC,2024-05-10 13:56:00
3,public\data\ftp\cnes\LT\2024\04\AC\LTAC2404.pa...,public\data\ftp\cnes\LT\2024\04\AC\LTAC2404.pa...,cnes,None,2024,4,AC,2024-05-24 13:44:00
4,public\data\ftp\cnes\LT\2024\05\AC\LTAC2405.pa...,public\data\ftp\cnes\LT\2024\05\AC\LTAC2405.pa...,cnes,None,2024,5,AC,2024-06-19 07:23:00


Cada linha é um arquivo mensal. Repare na coluna `name`: o nome do arquivo
começa com duas letras que indicam o **grupo** de informação. No CNES, por
exemplo, `LT` são os **leitos** e `ST` são os **estabelecimentos**.

Vamos ver quais grupos existem para o Acre em 2024:

In [5]:
# As duas primeiras letras do nome do arquivo indicam o grupo
grupos = sorted({nome.split("\\")[-1][:2] for nome in catalogo["name"]})

print("Grupos disponíveis:", grupos)

Grupos disponíveis: ['DC', 'EF', 'EP', 'EQ', 'GM', 'HB', 'IN', 'LT', 'PF', 'RC', 'SR', 'ST', 'cn']


## Passo 4 — Baixar o primeiro conjunto de dados

Agora que sabemos o que existe, vamos baixar os **leitos** (`LT`) do Acre em
dezembro de 2024. Escolhemos um estado pequeno e um único mês de propósito:
o download é rápido e serve para entender o formato.

> **Atenção ao `group`:** no CNES ele é obrigatório na prática. Sem ele, a
> biblioteca junta todos os grupos e devolve mais de 30 mil linhas e 362
> colunas — muito mais do que você precisa.

In [6]:
from pysus import cnes

leitos = cnes(state="AC", year=2024, month=12, group="LT", as_dataframe=True)

print(f"{len(leitos):,} linhas e {len(leitos.columns)} colunas")

276 linhas e 28 colunas


### Sempre confira se veio alguma coisa

Como a tabela vazia não gera erro, vale conferir antes de continuar:

In [7]:
if len(leitos) == 0:
    print("⚠ Nenhum dado veio. Confira no catálogo se este mês existe para este estado.")
else:
    print(f"✅ Dados carregados: {len(leitos):,} registros de leitos.")

✅ Dados carregados: 276 registros de leitos.


## Passo 5 — Olhar os dados

As primeiras linhas da tabela. Cada registro descreve os leitos de um
estabelecimento de saúde:

In [8]:
leitos.head()

,CNES,CODUFMUN,REGSAUDE,MICR_REG,DISTRSAN,DISTRADM,TPGESTAO,PF_PJ,CPF_CNPJ,NIV_DEP,CNPJ_MAN,ESFERA_A,ATIVIDAD,RETENCAO,NATUREZA,CLIENTEL,TP_UNID,TURNO_AT,NIV_HIER,TERCEIRO,TP_LEITO,CODLEITO,QT_EXIST,QT_CONTR,QT_SUS,QT_NSUS,COMPETEN,NAT_JUR
0,5701929,120001,001,,,,E,3,00000000000000,3,04034526000143,,04,,,03,15,06,,,2,33,3,0,3,0,202412,1023
1,5701929,120001,001,,,,E,3,00000000000000,3,04034526000143,,04,,,03,15,06,,,4,43,2,0,2,0,202412,1023
2,5701929,120001,001,,,,E,3,00000000000000,3,04034526000143,,04,,,03,15,06,,,5,45,2,0,2,0,202412,1023
3,2001020,120005,002,,,,E,3,04034526001034,3,04034526000143,,04,,,03,15,06,,,2,33,7,0,7,0,202412,1023
4,2001020,120005,002,,,,E,3,04034526001034,3,04034526000143,,04,,,03,15,06,,,4,43,2,0,2,0,202412,1023


As colunas mais úteis aqui são:

| Coluna | O que significa |
|---|---|
| `CNES` | Código do estabelecimento de saúde |
| `CODUFMUN` | Código IBGE do município |
| `TP_LEITO` | Tipo de leito (cirúrgico, clínico, UTI…) |
| `QT_EXIST` | Quantidade de leitos existentes |
| `QT_SUS` | Quantidade disponível para o SUS |

> Os nomes das colunas mudam de uma base para outra. Antes de usar qualquer
> coluna, vale olhar a lista com `leitos.columns` — foi por não fazer isso que
> muitos exemplos antigos quebraram.

In [9]:
list(leitos.columns)

['CNES',
 'CODUFMUN',
 'REGSAUDE',
 'MICR_REG',
 'DISTRSAN',
 'DISTRADM',
 'TPGESTAO',
 'PF_PJ',
 'CPF_CNPJ',
 'NIV_DEP',
 'CNPJ_MAN',
 'ESFERA_A',
 'ATIVIDAD',
 'RETENCAO',
 'NATUREZA',
 'CLIENTEL',
 'TP_UNID',
 'TURNO_AT',
 'NIV_HIER',
 'TERCEIRO',
 'TP_LEITO',
 'CODLEITO',
 'QT_EXIST',
 'QT_CONTR',
 'QT_SUS',
 'QT_NSUS',
 'COMPETEN',
 'NAT_JUR']

## Passo 6 — Uma primeira conta

Quantos leitos existem no total, e quantos estão disponíveis para o SUS?

Os números vêm como **texto** nos arquivos do DATASUS, então convertemos para
número antes de somar (`pd.to_numeric`). O `errors="coerce"` transforma em
vazio qualquer valor estranho, em vez de interromper tudo com um erro.

In [10]:
import pandas as pd

existentes = pd.to_numeric(leitos["QT_EXIST"], errors="coerce").sum()
sus = pd.to_numeric(leitos["QT_SUS"], errors="coerce").sum()

print(f"Leitos existentes no Acre (dez/2024): {existentes:,.0f}")
print(f"Disponíveis para o SUS:               {sus:,.0f}")
print(f"Proporção SUS:                        {sus / existentes:.1%}")

Leitos existentes no Acre (dez/2024): 1,807
Disponíveis para o SUS:               1,625
Proporção SUS:                        89.9%


## Pronto!

Você acabou de: instalar a biblioteca, consultar o catálogo, baixar dados reais
do DATASUS e fazer uma primeira análise.

### Para continuar

- **`02-descobrir-dados-disponiveis.ipynb`** — como achar qualquer base, estado
  e período disponíveis.
- Os notebooks das pastas `SINAN/`, `CNES/`, `SIM/` e `SINASC/` trazem análises
  completas, prontas para adaptar ao seu município ou estado.

### Três regras que evitam a maioria dos problemas

1. **Consulte o catálogo antes** (`list_files`) — nem todo mês existe.
2. **Confira `len(tabela)`** — tabela vazia não dá erro.
3. **Olhe `tabela.columns`** antes de usar uma coluna pelo nome.

---
*Notebook do projeto [PySusNoCode](https://github.com/cartaproale/PySusNoCode) —
um produto [Kraemer Academy](https://kraemeracademy.net).*

## Verificação de sanidade

Toda análise deste repositório termina conferindo o próprio resultado. Não é
formalidade: uma mudança no catálogo do DATASUS já fez notebooks devolverem o
Brasil inteiro rotulado como um estado, **sem erro nenhum**. Falhar alto é
incômodo; acertar o formato e errar o número vira decisão errada.

In [11]:
print("Verificações\n")
falhas = []


def conferir(numero, descricao, condicao, detalhe=""):
    marca = "confere" if condicao else "ATENÇÃO"
    print(f"{numero}. {descricao}{(' — ' + detalhe) if detalhe else ''}: {marca}")
    if not condicao:
        falhas.append(descricao)

conferir(1, "O download trouxe leitos", len(leitos) > 0, f"{len(leitos):,} registros")
conferir(2, "Leitos SUS não passam do total de leitos",
         sus <= existentes, f"{sus:,.0f} de {existentes:,.0f}")
conferir(3, "Ordem de grandeza plausível para um estado",
         existentes > 100, f"{existentes:,.0f} leitos")
conferir(4, "A proporção de leitos SUS faz sentido (entre 20% e 100%)",
         20 <= sus / existentes * 100 <= 100, f"{sus / existentes * 100:.0f}% são SUS")

print()
if falhas:
    print("ATENÇÃO: revise antes de usar estes números —", falhas)
else:
    print("Tudo confere.")

Verificações
1. O download trouxe leitos — 276 registros: confere
2. Leitos SUS não passam do total de leitos — 1,625 de 1,807: confere
3. Ordem de grandeza plausível para um estado — 1,807 leitos: confere
4. A proporção de leitos SUS faz sentido (entre 20% e 100%) — 90% são SUS: confere
Tudo confere.
